# 02 流病視覺化：5 張關鍵圖表

用松柏護理之家退伍軍人症 line list，學會 matplotlib / seaborn / plotly 三大繪圖套件。

| 圖表 | 使用套件 | 觀察重點 |
|------|---------|----------|
| 流行曲線 | matplotlib | 傳播模式（共同暴露源 vs 持續傳播） |
| 年齡分布 | seaborn | 年齡是否為危險因子 |
| 翼區侵襲率 | seaborn | 空間聚集線索 |
| 嚴重度×共病 | seaborn heatmap | 多因子交互 |
| 互動分層曲線 | plotly | 各樓層流行高峰比較 |

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- 資料準備（與前一堂課相同） ---
# 這個 cell 做三件事：
#   1) 匯入繪圖套件（matplotlib, seaborn, plotly）
#   2) 設定全域圖表風格（字型、解析度）
#   3) 讀入資料並建立衍生變項
#
# 💡 三大繪圖套件的角色：
#   matplotlib → 底層引擎，像蓋房子自己砌磚（最大彈性）
#   seaborn    → matplotlib 的高級包裝，像買預製屋（寫少做多，統計圖特別方便）
#   plotly     → 互動式圖表引擎，滑鼠可懸停、縮放（適合簡報和網頁）

import pathlib

import pandas as pd
import matplotlib.pyplot as plt       # plt = matplotlib 的慣用縮寫
import matplotlib.font_manager as fm  # fm = 字型管理器（處理中文字型）
import seaborn as sns                 # sns = seaborn 的慣用縮寫
import plotly.express as px           # px = plotly 快速繪圖介面
import plotly.io as pio               # pio = plotly 的輸入輸出設定

# -- 全域圖表風格設定 --
# plt.style.use("ggplot") → 套用 ggplot 風格（淡灰背景 + 白色格線，學術論文常用）
# plt.rcParams["figure.dpi"] = 150 → 提高解析度（預設 100 在螢幕上太模糊）
# 💡 rcParams = "runtime configuration parameters"，控制 matplotlib 的所有預設值
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

# -- CJK font setup (避免中文標籤顯示為方框 □□□) --
# 問題：matplotlib 預設只認英文字型，中文字會變成「豆腐塊」
# 解法：手動掃描系統字型目錄，註冊所有 CJK（中日韓）字型
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

# 設定中文字型候選清單（按優先順序嘗試）
plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False  # 防止負號顯示為方塊

# Plotly: 確保在靜態建置（jupyter-book build）時也能輸出互動圖
pio.renderers.default = "notebook"

# --- 讀入資料 ---
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")

# 日期轉換（與 Step 3 相同）
date_cols = [
    "facility_admission_date", "symptom_onset_date",
    "hospitalization_date", "death_date", "notification_date",
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# 衍生變項（與 Step 4 相同）
comorbidity_cols = [
    "comorbidity_chf", "comorbidity_dm",
    "comorbidity_cancer", "comorbidity_copd", "immunosuppressed",
]
df["n_comorbidities"] = df[comorbidity_cols].sum(axis=1)
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["age_group"] = pd.cut(
    df["age"], bins=[59, 69, 79, 89, 100],
    labels=["60-69", "70-79", "80-89", "90+"],
)

# 只取感染者（畫流行曲線等用）
# df[df["infected"] == 1] → 布林篩選，只保留 infected=1 的列
# .copy() → 建立獨立副本，避免修改時產生 SettingWithCopyWarning
cases = df[df["infected"] == 1].copy()
print(f"全體：{len(df)} 人，感染者：{len(cases)} 人")

## 1) 流行曲線 — Epidemic Curve（matplotlib）

最經典的流病圖表。X 軸是發病日，Y 軸是每日新增病例。
曲線形狀可推斷傳播模式：尖峰 → 共同暴露源；拖尾 → 持續傳播。

### 繪製要點（CDC / ECDC 規範）

流行曲線本質上是**直方圖（histogram）**，不是一般的長條圖（bar chart）：

- **相鄰長條不留間隙**：X 軸是連續時間軸，長條之間不應有空隙（`width=1.0`）
- **補齊沒有病例的日期**：即使某天 0 例也要佔位（用 `reindex` 填 0），否則 X 軸間距失真
- **顯示爆發前背景期**：包含疫情爆發前 1–2 個潛伏期的日期，讓讀者看到疫情何時偏離背景值
- **標題要能獨立閱讀**：包含疾病名稱、地點、時間範圍
- **X 軸**：標示「發病日期（Date of Symptom Onset）」——明確說明時間基準
- **Y 軸**：標示「病例數（Number of Cases）」——必須是整數刻度，從 0 開始，不截斷
- **隱藏格線**：減少視覺干擾，去除上方和右方邊框
- **個案分類用顏色區分**：確診 vs 疑似須用不同顏色並附圖例
- **不在長條上標數字**：避免數位與類比資訊互相干擾

In [ ]:
# --- 流行曲線繪製 ---
# matplotlib 繪圖的核心模式：fig, ax = plt.subplots()
#   fig = figure（整張圖紙）
#   ax  = axes（圖紙上的繪圖區域）
#   所有繪圖指令都對 ax 操作（ax.bar, ax.set_title, ...）
#
# 💡 為什麼不用 plt.plot()？
#   plt.plot() 是「簡易模式」，一張圖可以用
#   fig, ax 是「專業模式」，可以在同一張圖紙上放多個子圖
#   學術論文和疫調報告幾乎都用 fig, ax 模式

import matplotlib.dates as mdates  # 日期格式化工具

# 1) 計算每日病例數
daily = cases.groupby("symptom_onset_date").size().rename("cases")

# 2) 補齊完整日期範圍（包含爆發前 3 天，顯示背景期）
#    pd.date_range() → 產生連續日期序列
#    .reindex(fill_value=0) → 沒有病例的日期補 0（不能留空！）
date_range = pd.date_range(
    daily.index.min() - pd.Timedelta(days=3),
    daily.index.max() + pd.Timedelta(days=1),
    freq="D",
)
daily = daily.reindex(date_range, fill_value=0)

# 3) 繪製長條圖
fig, ax = plt.subplots(figsize=(10, 4))  # figsize=(寬, 高) 單位是英吋
ax.bar(
    daily.index, daily.values,
    width=1.0,                         # 寬度=1（天），長條緊密貼合
    color="#2c7fb8",                    # 長條填色
    edgecolor="white", linewidth=0.5,  # 白色邊框讓長條可區分
)

# 4) 標題和軸標籤
# 💡 好標題 = 疾病 + 地點 + 時間，讓圖片單獨看也能理解
ax.set_title(
    "松柏護理之家退伍軍人症流行曲線，依發病日，2026 年 1 月",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("發病日期（Date of Symptom Onset）")
ax.set_ylabel("病例數（Number of Cases）")

# 5) 日期格式化
# DateFormatter("%m/%d") → 顯示「月/日」格式
# DayLocator(interval=2) → 每隔 2 天標一個刻度
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45)  # 日期標籤旋轉 45 度，避免重疊

# 6) 座標軸微調
ax.set_xlim(
    daily.index.min() - pd.Timedelta(hours=12),
    daily.index.max() + pd.Timedelta(hours=12),
)
ax.set_ylim(bottom=0)                              # Y 軸從 0 開始
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))  # Y 軸只顯示整數

# 7) CDC 風格：隱藏格線、去除上右邊框（讓圖表更乾淨）
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()  # 自動調整邊距，防止標籤被裁切
plt.show()

### 經典方格式流行曲線（Unit Chart）

教科書和 CDC 疫調報告中常見的方格式（stacked squares）流行曲線——每個小方格代表一個病例，堆疊形成柱狀。這裡我們用顏色區分確診與疑似個案。

> 💡 方格式特別適合**小規模群聚**（數十至百餘例）。病例數太大時方格會太小，改用標準直方圖更合適。

In [ ]:
# --- 方格式流行曲線（Unit Chart）---
# 每個小方格 = 一個病例，用顏色區分確診 vs 疑似
# 適合小規模群聚（數十～百餘例），大規模疫情用標準直方圖
#
# 技術要點：
#   plt.Rectangle(xy, width, height) → 畫一個矩形
#   ax.add_patch(rect) → 把矩形加到圖上
#   mdates.date2num(date) → 把日期轉成 matplotlib 的數字座標

# 準備每日 confirmed / probable 的病例數
# .unstack(fill_value=0) → 把 case_classification 從「列」轉成「欄」
daily_class = (
    cases.groupby(["symptom_onset_date", "case_classification"])
    .size()
    .unstack(fill_value=0)
)
daily_class = daily_class.reindex(date_range, fill_value=0)
colors_map = {"confirmed": "#2c7fb8", "probable": "#a6bddb"}

fig, ax = plt.subplots(figsize=(10, 5))
box_size = 1.0

for date in daily_class.index:
    x = mdates.date2num(date)      # 日期 → 數字座標
    j = 0                          # j = 目前堆疊高度（從 0 開始往上疊）
    for cls in ["confirmed", "probable"]:
        count = daily_class.at[date, cls] if cls in daily_class.columns else 0
        for _ in range(int(count)):  # 每個病例畫一個方格
            rect = plt.Rectangle(
                (x - box_size / 2, j * box_size),  # 左下角座標
                box_size, box_size,                  # 寬、高
                facecolor=colors_map[cls],
                edgecolor="white", linewidth=0.8,
            )
            ax.add_patch(rect)
            j += 1

# 座標軸設定
ax.set_xlim(
    mdates.date2num(daily_class.index.min()) - 1.5,
    mdates.date2num(daily_class.index.max()) + 1.5,
)
y_max = daily_class.sum(axis=1).max()
ax.set_ylim(0, y_max + 1)
ax.set_aspect("equal")  # 讓方格是正方形（寬=高）

ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

ax.set_title(
    "松柏護理之家退伍軍人症流行曲線 — 方格式（依個案分類）",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("發病日期（Date of Symptom Onset）")
ax.set_ylabel("病例數（Number of Cases）")

# 手動圖例（因為我們用 add_patch 畫的，matplotlib 不會自動產生圖例）
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#2c7fb8", edgecolor="white", label="確診（Confirmed）"),
    Patch(facecolor="#a6bddb", edgecolor="white", label="疑似（Probable）"),
]
ax.legend(handles=legend_elements, loc="upper left", frameon=False)

ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## 2) 年齡分布：感染 vs 未感染（seaborn）

把全體 280 人的年齡分布疊起來，看感染者是否集中在特定年齡層。

In [ ]:
# --- 年齡分布圖（seaborn）---
# seaborn 是 matplotlib 的高級包裝，用一行就能畫出漂亮的統計圖
#
# sns.histplot() 參數說明：
#   data=df        → 指定資料來源（整個 DataFrame）
#   x="age"        → X 軸用 age 欄位
#   hue="infected" → 用 infected 欄位分色（0=未感染, 1=感染）
#   hue_order=[1,0]→ 圖例順序：先顯示感染者
#   bins=15        → 分成 15 個區間（太少看不到細節，太多太碎）
#   multiple="stack" → 堆疊模式（感染者疊在未感染者上面）
#   palette={1:"紅", 0:"灰"} → 自訂顏色對應
#   ax=ax          → 畫在哪個 axes 上
#
# 💡 seaborn vs matplotlib 的差異：
#   matplotlib: ax.bar(x, y, color=...) — 你要自己準備 x, y 數據
#   seaborn: sns.histplot(data=df, x="age") — 直接丟 DataFrame，它幫你算

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(
    data=df, x="age", hue="infected", hue_order=[1, 0], bins=15,
    multiple="stack", palette={1: "#e34a33", 0: "#cccccc"}, ax=ax,
)
ax.set_title("年齡分布：感染 vs 未感染")
ax.set_xlabel("年齡")
ax.set_ylabel("人數")
ax.legend(title="感染", labels=["感染", "未感染"])
plt.tight_layout()
plt.show()

## 3) 各翼區侵襲率長條圖（seaborn）

不能直接比病例數——要除以分母（住民數）才公平。
侵襲率異常偏高的翼區可能有共同暴露源（例如淋浴設備）。

In [ ]:
# --- 各翼區侵襲率長條圖（seaborn）---
# sns.barplot() 參數說明：
#   data=wing_stats → 指定資料來源
#   x="label"       → X 軸用翼區標籤（如 "1A", "2B"）
#   y="attack_rate_pct" → Y 軸用侵襲率百分比
#   hue="label"     → 每個翼區不同顏色
#   palette="YlOrRd"→ 黃→橘→紅漸層色盤（值越高越紅）
#
# 💡 為什麼用侵襲率而不是病例數？
#   A 翼 50 人中 30 人感染（60%）vs B 翼 100 人中 30 人感染（30%）
#   病例數相同（都是 30），但 A 翼的風險高一倍！
#   分母很重要 → 永遠要除以分母

# 計算翼區統計
wing_stats = (
    df.groupby(["floor", "wing"])
    .agg(residents=("case_id", "size"), infected=("infected", "sum"))
    .reset_index()
)
wing_stats["attack_rate_pct"] = (
    wing_stats["infected"] / wing_stats["residents"] * 100
).round(1)
wing_stats["label"] = wing_stats["floor"].astype(str) + wing_stats["wing"]
wing_stats = wing_stats.sort_values("attack_rate_pct", ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(
    data=wing_stats, x="label", y="attack_rate_pct",
    hue="label", palette="YlOrRd", legend=False, ax=ax,
)
ax.set_title("各翼區侵襲率比較")
ax.set_xlabel("翼區")
ax.set_ylabel("侵襲率 (%)")

# 在長條上方標數字 — 讓讀者不用看 Y 軸就能讀到精確值
# itertuples() → 把每列轉成 tuple，比 iterrows() 快
for i, row in enumerate(wing_stats.itertuples()):
    ax.text(i, row.attack_rate_pct + 1, f"{row.attack_rate_pct}%",
            ha="center", fontsize=10)

plt.tight_layout()
plt.show()

## 4) 嚴重度 × 共病數熱力圖（seaborn）

共病越多的人是否更容易重症？用熱力圖交叉比較。

In [ ]:
# --- 嚴重度 × 共病數熱力圖（seaborn）---
# 熱力圖 = 用顏色深淺表示數值大小的二維表格
# 適合觀察「兩個類別變項的交叉關係」
#
# sns.heatmap() 參數說明：
#   heat_data     → 2D 表格（列=嚴重度, 欄=共病數）
#   annot=True    → 在每個格子裡顯示數字
#   fmt="d"       → 數字格式為整數（d=digit）
#   cmap="YlOrRd" → 色盤：黃→橘→紅（數字越大越紅）
#
# 資料準備流程：
#   1) 篩選有症狀的感染者（排除 not_ill 和 asymptomatic）
#   2) groupby 兩個欄位 → .size() 計算每組人數
#   3) .unstack() 把共病數從列變成欄（變成二維表格）
#   4) .reindex() 確保嚴重度按 mild → moderate → severe 排序

severity_order = ["mild", "moderate", "severe"]
heat_data = (
    cases[cases["clinical_severity"].isin(severity_order)]
    .groupby(["clinical_severity", "n_comorbidities"])
    .size()
    .unstack(fill_value=0)
    .reindex(severity_order)  # 確保列順序
)

fig, ax = plt.subplots(figsize=(8, 3.5))
sns.heatmap(heat_data, annot=True, fmt="d", cmap="YlOrRd", ax=ax)
ax.set_title("臨床嚴重度 × 共病數")
ax.set_xlabel("共病數")
ax.set_ylabel("嚴重度")
plt.tight_layout()
plt.show()

## 5) 互動式分層流行曲線（Plotly）

用 Plotly 把流行曲線按樓層上色，滑鼠懸停可看數值。Plotly 的互動式圖表同樣需要遵循 CDC 流行曲線繪製規範：無間隙（`bargap=0`）、描述性標題、隱藏格線、Y 軸從 0 開始。

觀察：三個樓層的流行高峰是否同步？如果不同步，代表什麼？

In [ ]:
# --- 互動式分層流行曲線（Plotly）---
# Plotly 的特色：滑鼠懸停可看數值、可縮放、可匯出為 HTML
#
# px.bar() 參數說明（與 matplotlib 的差異）：
#   x="欄位名"     → 直接指定欄位名（不用先計算）
#   y="欄位名"     → 同上
#   color="floor"  → 用 floor 欄位分色（自動產生圖例）
#   barmode="stack" → 堆疊模式
#   color_discrete_sequence=[...] → 自訂顏色序列
#   title="..."    → 標題（一個參數搞定）
#   labels={...}   → 自訂軸標籤（用字典對應）
#
# 💡 Plotly vs matplotlib 的核心差異：
#   matplotlib: 「命令式」— 一步步告訴它怎麼畫（set_title, set_xlabel...）
#   plotly:     「宣告式」— 告訴它你要什麼，它自己畫（一個函式搞定）

import plotly.express as px
import plotly.graph_objects as go

# 依樓層分層，並補齊完整日期範圍
daily_floor = (
    cases.groupby(["symptom_onset_date", "floor"])
    .size()
    .rename("cases")
    .reset_index()
)
daily_floor["floor"] = daily_floor["floor"].astype(str) + "F"

# 補齊所有日期 × 樓層組合（含 0 例的天數）
all_dates = pd.date_range(
    cases["symptom_onset_date"].min() - pd.Timedelta(days=3),
    cases["symptom_onset_date"].max() + pd.Timedelta(days=1),
    freq="D",
)
all_floors = sorted(daily_floor["floor"].unique())
full_idx = pd.MultiIndex.from_product(
    [all_dates, all_floors], names=["symptom_onset_date", "floor"]
)
daily_floor = (
    daily_floor.set_index(["symptom_onset_date", "floor"])
    .reindex(full_idx, fill_value=0)
    .reset_index()
)

fig = px.bar(
    daily_floor,
    x="symptom_onset_date", y="cases", color="floor",
    barmode="stack",
    color_discrete_sequence=["#2c7fb8", "#41ae76", "#fe9929"],
    title="松柏護理之家退伍軍人症流行曲線，依樓層與發病日，2026 年 1 月",
    labels={"symptom_onset_date": "發病日期（Date of Symptom Onset）",
            "cases": "病例數（Number of Cases）",
            "floor": "樓層"},
)

# fig.update_layout() → Plotly 的「微調」方法（類似 matplotlib 的 ax.set_xxx）
fig.update_layout(
    bargap=0,                              # 長條之間無間隙（CDC 規範）
    xaxis=dict(showgrid=False),            # 隱藏垂直格線
    yaxis=dict(showgrid=False, rangemode="tozero"),  # Y 軸從 0 開始
    plot_bgcolor="white",                  # 白色背景
    # 圖例放在圖表上方（水平排列）
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig.show()

## 6) 圖表輸出——疫調報告與期刊投稿

畫好的圖要怎麼存檔？不同用途有不同的格式和解析度要求：

| 用途 | 格式 | DPI | 說明 |
|------|------|-----|------|
| 疫調報告 / 簡報 | PNG | 150–300 | 點陣圖，檔案小，適合 Word/PPT |
| 期刊投稿 | PDF / SVG | 向量 | 無限放大不失真，排版首選 |
| 網頁 / 互動 | HTML | — | Plotly 專用，保留互動功能 |

### 期刊投稿規格速查

| 期刊 | 解析度 | 單欄寬度 | 字型 | 特殊要求 |
|------|--------|---------|------|---------|
| NEJM | ≥1000 DPI | 8.9 cm | Arial | 色盲友善 |
| Lancet | ≥300 DPI | 8.5 cm | Arial | 純黑白可辨識 |
| JAMA | ≥350 DPI | 8.4 cm | Arial | EPS 或 PDF |

In [ ]:
# --- 圖表輸出範例 ---

# ===== matplotlib 輸出 =====
# fig.savefig() 參數說明：
#   "檔名.png"         → 輸出檔案路徑（副檔名決定格式：.png/.pdf/.svg）
#   dpi=300            → 解析度（疫調報告用 300，期刊用 ≥600）
#   bbox_inches="tight"→ 自動裁切多餘空白（超重要！不加會留很大白邊）
#   facecolor="white"  → 背景色設為白色（預設可能是透明）

# 重新畫一次流行曲線作為輸出示範
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(daily.index, daily.values, width=1.0,
       color="#2c7fb8", edgecolor="white", linewidth=0.5)
ax.set_title("松柏護理之家退伍軍人症流行曲線，依發病日，2026 年 1 月",
             fontsize=13, fontweight="bold")
ax.set_xlabel("發病日期（Date of Symptom Onset）")
ax.set_ylabel("病例數（Number of Cases）")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45)
ax.set_ylim(bottom=0)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# 輸出為 PNG（疫調報告用）
fig.savefig("epi_curve_report.png", dpi=300, bbox_inches="tight", facecolor="white")

# 輸出為 PDF（期刊投稿用，向量圖不失真）
fig.savefig("epi_curve_journal.pdf", bbox_inches="tight", facecolor="white")

print("✅ 已輸出：epi_curve_report.png (300 DPI) + epi_curve_journal.pdf (向量圖)")
plt.show()

# ===== Plotly 輸出 =====
# fig.write_html("檔名.html")  → 保留互動功能，適合網頁報告
# fig.write_image("檔名.png")  → 靜態圖片（需安裝 kaleido 套件）
# 💡 提示：write_image 需要 `uv add kaleido`

## 小結

這堂課你學會了 5 種流病常用圖表 + 圖表輸出：

| 圖表 | 套件 | 觀察重點 |
|------|------|----------|
| 流行曲線 | matplotlib | 峰值時間、上升/下降速度 → 傳播模式 |
| 年齡分布 | seaborn | 感染者是否集中在特定年齡層 |
| 翼區長條圖 | seaborn | 哪些翼區侵襲率異常偏高 → 空間線索 |
| 嚴重度×共病 | seaborn heatmap | 共病多的人是否更容易重症 |
| 互動分層曲線 | plotly | 各樓層的流行高峰是否同步 |

### 三大繪圖套件速查

| 套件 | 適合場景 | 核心語法 |
|------|---------|---------|
| **matplotlib** | 完全客製化、期刊投稿 | `fig, ax = plt.subplots()` → `ax.bar()` → `ax.set_title()` |
| **seaborn** | 統計圖表（直方圖、熱力圖） | `sns.histplot(data=df, x="age", hue="infected")` |
| **plotly** | 互動式圖表、網頁報告 | `px.bar(df, x="date", y="cases", color="floor")` |

### 圖表輸出速查

| 方法 | 格式 | 用途 |
|------|------|------|
| `fig.savefig("圖.png", dpi=300, bbox_inches="tight")` | PNG | 疫調報告 / 簡報 |
| `fig.savefig("圖.pdf", bbox_inches="tight")` | PDF | 期刊投稿（向量圖） |
| `fig.write_html("圖.html")` | HTML | 互動式網頁報告（Plotly） |

下一章（Ch03 描述性統計），我們會把這些觀察量化——計算 2×2 表、卡方檢定、風險比。